In [30]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    cleaned_file_path = Path("output") / "beauty_sales_cleaned_final.csv"

    if not cleaned_file_path.exists():
        raise FileNotFoundError(f"{cleaned_file_path} not found. Run Phase 3 Cell 8 first.")

    df = pd.read_csv(cleaned_file_path)

    if df.empty:
        raise SystemExit("ERROR: Cleaned dataset is empty. Check Phase 3 output.")

    # Convert date again to be safe after CSV load
    if "date" not in df.columns:
        raise SystemExit("ERROR: 'date' column is missing from cleaned dataset.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    print("Cleaned data loaded successfully.")
    print("Shape:", df.shape)
    print("Date range:", df["date"].min(), "to", df["date"].max())
    print("\nColumns:")
    print(df.columns.tolist())

    display(df.head())

except FileNotFoundError as e:
    raise SystemExit(f"ERROR: {e}")

except Exception as e:
    raise SystemExit(f"ERROR: Could not load cleaned dataset — {e}")

Cleaned data loaded successfully.
Shape: (2865, 13)
Date range: 2026-02-01 00:00:00 to 2026-05-18 00:00:00

Columns:
['order_id', 'date', 'channel', 'sku', 'product_name', 'category', 'customer_type', 'orders', 'units_sold', 'gross_revenue', 'discount_amount', 'net_revenue', 'returns']


,order_id,date,channel,sku,product_name,category,customer_type,orders,units_sold,gross_revenue,discount_amount,net_revenue,returns
0,ORD10001,2026-02-01,Amazon,FACE002,Skin Blur Primer,Face,New,4,6,5394,761.09,4632.91,0
1,ORD10002,2026-02-01,Flipkart,LIP002,Hydra Gloss Tint,Lips,New,3,4,2396,155.50,2240.50,0
2,ORD10003,2026-02-01,Flipkart,LIP001,Velvet Matte Lipstick,Lips,Returning,4,4,2796,259.83,2536.17,0
3,ORD10004,2026-02-01,Flipkart,SKIN001,Dew Fix Setting Spray,Skincare,Returning,3,4,2996,452.02,2543.98,0
4,ORD10005,2026-02-01,Amazon,FACE003,Cream Blush,Face,New,8,8,5192,417.90,4774.10,0


In [31]:
try:
    if df["date"].isna().all():
        raise SystemExit("ERROR: All values in 'date' are invalid. Cannot do weekly analysis.")

    # Drop rows with invalid dates for weekly analysis
    df_analysis = df.dropna(subset=["date"]).copy()

    if df_analysis.empty:
        raise SystemExit("ERROR: No valid dated rows found after filtering invalid dates.")

    latest_date = df_analysis["date"].max().normalize()

    # Find the last completed Sunday
    if latest_date.weekday() == 6:
        last_sunday = latest_date
    else:
        last_sunday = latest_date - pd.Timedelta(days=latest_date.weekday() + 1)

    current_week_start = last_sunday - pd.Timedelta(days=6)
    current_week_end = last_sunday

    previous_week_start = current_week_start - pd.Timedelta(days=7)
    previous_week_end = current_week_end - pd.Timedelta(days=7)

    current_week_df = df_analysis[
        (df_analysis["date"] >= current_week_start) &
        (df_analysis["date"] <= current_week_end)
    ].copy()

    previous_week_df = df_analysis[
        (df_analysis["date"] >= previous_week_start) &
        (df_analysis["date"] <= previous_week_end)
    ].copy()

    print("Weekly date windows created successfully.\n")

    print("Latest date in dataset     :", latest_date.date())
    print("Current week start         :", current_week_start.date())
    print("Current week end           :", current_week_end.date())
    print("Previous week start        :", previous_week_start.date())
    print("Previous week end          :", previous_week_end.date())

    print("\nCurrent week rows          :", len(current_week_df))
    print("Previous week rows         :", len(previous_week_df))

    if current_week_df.empty:
        print("\nWARNING: Current week DataFrame is empty.")

    if previous_week_df.empty:
        print("WARNING: Previous week DataFrame is empty.")

except Exception as e:
    raise SystemExit(f"ERROR: Could not create weekly slices — {e}")

Weekly date windows created successfully.

Latest date in dataset     : 2026-05-18
Current week start         : 2026-05-11
Current week end           : 2026-05-17
Previous week start        : 2026-05-04
Previous week end          : 2026-05-10

Current week rows          : 168
Previous week rows         : 187


In [32]:
try:
    def get_basic_kpis(dataframe):
        revenue = dataframe["net_revenue"].sum() if "net_revenue" in dataframe.columns else 0
        orders = dataframe["orders"].sum() if "orders" in dataframe.columns else 0
        aov = revenue / orders if orders != 0 else 0
        return {
            "revenue": round(float(revenue), 2),
            "orders": int(orders),
            "aov": round(float(aov), 2)
        }

    current_kpis = get_basic_kpis(current_week_df)
    previous_kpis = get_basic_kpis(previous_week_df)

    def get_pct_change(current, previous):
        if previous == 0:
            return None
        return round(((current - previous) / previous) * 100, 2)

    kpi_summary = {
        "current_revenue": current_kpis["revenue"],
        "previous_revenue": previous_kpis["revenue"],
        "revenue_pct_change": get_pct_change(current_kpis["revenue"], previous_kpis["revenue"]),

        "current_orders": current_kpis["orders"],
        "previous_orders": previous_kpis["orders"],
        "orders_pct_change": get_pct_change(current_kpis["orders"], previous_kpis["orders"]),

        "current_aov": current_kpis["aov"],
        "previous_aov": previous_kpis["aov"],
        "aov_pct_change": get_pct_change(current_kpis["aov"], previous_kpis["aov"])
    }

    print("Core KPI summary:")
    print(kpi_summary)

except Exception as e:
    raise SystemExit(f"ERROR: Could not calculate core KPIs — {e}")

Core KPI summary:
{'current_revenue': 446285.89, 'previous_revenue': 477068.46, 'revenue_pct_change': -6.45, 'current_orders': 594, 'previous_orders': 632, 'orders_pct_change': -6.01, 'current_aov': 751.32, 'previous_aov': 754.86, 'aov_pct_change': -0.47}


In [33]:
try:
    def get_customer_mix(dataframe):
        result = {}

        if "customer_type" not in dataframe.columns:
            result["new_customers"] = 0
            result["returning_customers"] = 0
            result["pct_new"] = 0
            result["pct_returning"] = 0
            return result

        customer_counts = dataframe["customer_type"].value_counts(dropna=False)

        new_count = int(customer_counts.get("New", 0))
        returning_count = int(customer_counts.get("Returning", 0))
        total = new_count + returning_count

        result["new_customers"] = new_count
        result["returning_customers"] = returning_count
        result["pct_new"] = round((new_count / total) * 100, 2) if total != 0 else 0
        result["pct_returning"] = round((returning_count / total) * 100, 2) if total != 0 else 0

        return result

    customer_mix_summary = get_customer_mix(current_week_df)

    print("Customer mix summary:")
    print(customer_mix_summary)

except Exception as e:
    raise SystemExit(f"ERROR: Could not calculate customer mix — {e}")

Customer mix summary:
{'new_customers': 65, 'returning_customers': 103, 'pct_new': 38.69, 'pct_returning': 61.31}


In [34]:
try:
    required_cols = {"channel", "net_revenue", "orders"}

    if not required_cols.issubset(current_week_df.columns):
        raise SystemExit("ERROR: Required columns for channel breakdown are missing.")

    channel_summary = (
        current_week_df
        .groupby("channel", dropna=False)
        .agg(
            revenue=("net_revenue", "sum"),
            orders=("orders", "sum")
        )
        .reset_index()
        .sort_values("revenue", ascending=False)
        .reset_index(drop=True)
    )

    total_channel_revenue = channel_summary["revenue"].sum()
    channel_summary["revenue_pct"] = np.where(
        total_channel_revenue != 0,
        round((channel_summary["revenue"] / total_channel_revenue) * 100, 2),
        0
    )

    print("Channel summary created successfully.")
    display(channel_summary)

except Exception as e:
    raise SystemExit(f"ERROR: Could not create channel breakdown — {e}")

Channel summary created successfully.


,channel,revenue,orders,revenue_pct
0,Website,102612.82,143,22.99
1,Nykaa,78863.79,114,17.67
2,Amazon,78024.31,104,17.48
3,Myntra,74520.43,90,16.70
4,Flipkart,59350.78,75,13.30
5,Instagram Shop,52913.76,68,11.86


In [35]:
try:
    product_col = "product_name" if "product_name" in current_week_df.columns else "sku"

    if product_col not in current_week_df.columns or "net_revenue" not in current_week_df.columns:
        raise SystemExit("ERROR: Required columns for top product analysis are missing.")

    top_products_summary = (
        current_week_df
        .groupby(product_col, dropna=False)
        .agg(
            revenue=("net_revenue", "sum"),
            orders=("orders", "sum") if "orders" in current_week_df.columns else ("net_revenue", "size")
        )
        .reset_index()
        .sort_values("revenue", ascending=False)
        .head(4)
        .reset_index(drop=True)
    )

    print("Top 4 products created successfully.")
    display(top_products_summary)

except Exception as e:
    raise SystemExit(f"ERROR: Could not create top products summary — {e}")

Top 4 products created successfully.


,product_name,revenue,orders
0,Skin Blur Primer,86727.05,94
1,Silk Finish Foundation,81620.22,79
2,Cream Blush,62747.23,93
3,Radiance Compact,56805.98,69


In [36]:
try:
    if "date" not in current_week_df.columns or "net_revenue" not in current_week_df.columns:
        raise SystemExit("ERROR: Required columns for anomaly detection are missing.")

    daily_revenue = (
        current_week_df
        .groupby("date", dropna=False)
        .agg(daily_revenue=("net_revenue", "sum"))
        .reset_index()
        .sort_values("date")
        .reset_index(drop=True)
    )

    mean_revenue = daily_revenue["daily_revenue"].mean()
    std_revenue = daily_revenue["daily_revenue"].std()

    if pd.isna(std_revenue) or std_revenue == 0:
        daily_revenue["z_score"] = 0
        daily_revenue["is_anomaly"] = False
    else:
        daily_revenue["z_score"] = (daily_revenue["daily_revenue"] - mean_revenue) / std_revenue
        daily_revenue["is_anomaly"] = daily_revenue["z_score"].abs() > 2

    anomaly_days = daily_revenue[daily_revenue["is_anomaly"]].copy()

    print("Daily revenue table:")
    display(daily_revenue)

    print("\nAnomaly days:")
    display(anomaly_days)

except Exception as e:
    raise SystemExit(f"ERROR: Could not detect anomaly days — {e}")

Daily revenue table:


,date,daily_revenue,z_score,is_anomaly
0,2026-05-11,52120.56,-0.500502,False
1,2026-05-12,84737.25,0.902620,False
2,2026-05-13,39570.46,-1.040389,False
3,2026-05-14,57904.70,-0.251677,False
4,2026-05-15,57967.90,-0.248958,False
5,2026-05-16,48152.31,-0.671210,False
6,2026-05-17,105832.71,1.810116,False



Anomaly days:


,date,daily_revenue,z_score,is_anomaly


In [37]:
import json

try:
    weekly_summary = {
        "week_start": str(current_week_start.date()),
        "week_end": str(current_week_end.date()),
        "previous_week_start": str(previous_week_start.date()),
        "previous_week_end": str(previous_week_end.date()),

        "total_revenue": kpi_summary["current_revenue"],
        "previous_total_revenue": kpi_summary["previous_revenue"],
        "revenue_pct_change": kpi_summary["revenue_pct_change"],

        "total_orders": kpi_summary["current_orders"],
        "previous_total_orders": kpi_summary["previous_orders"],
        "orders_pct_change": kpi_summary["orders_pct_change"],

        "aov": kpi_summary["current_aov"],
        "previous_aov": kpi_summary["previous_aov"],
        "aov_pct_change": kpi_summary["aov_pct_change"],

        "new_customers": customer_mix_summary["new_customers"],
        "returning_customers": customer_mix_summary["returning_customers"],
        "pct_new_customers": customer_mix_summary["pct_new"],
        "pct_returning_customers": customer_mix_summary["pct_returning"],

        "anomaly_days_count": int(anomaly_days.shape[0])
    }

    print("Weekly summary object created successfully.")
    print(json.dumps(weekly_summary, indent=2))

except Exception as e:
    raise SystemExit(f"ERROR: Could not build weekly summary object — {e}")

Weekly summary object created successfully.
{
  "week_start": "2026-05-11",
  "week_end": "2026-05-17",
  "previous_week_start": "2026-05-04",
  "previous_week_end": "2026-05-10",
  "total_revenue": 446285.89,
  "previous_total_revenue": 477068.46,
  "revenue_pct_change": -6.45,
  "total_orders": 594,
  "previous_total_orders": 632,
  "orders_pct_change": -6.01,
  "aov": 751.32,
  "previous_aov": 754.86,
  "aov_pct_change": -0.47,
  "new_customers": 65,
  "returning_customers": 103,
  "pct_new_customers": 38.69,
  "pct_returning_customers": 61.31,
  "anomaly_days_count": 0
}


In [38]:
try:
    output_dir = Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)

    weekly_summary_json_path = output_dir / "weekly_summary.json"
    weekly_channel_csv_path = output_dir / "weekly_channel_breakdown.csv"
    weekly_top_products_csv_path = output_dir / "weekly_top_products.csv"
    weekly_daily_revenue_csv_path = output_dir / "weekly_daily_revenue.csv"
    weekly_anomaly_days_csv_path = output_dir / "weekly_anomaly_days.csv"

    with open(weekly_summary_json_path, "w", encoding="utf-8") as f:
        json.dump(weekly_summary, f, indent=2)

    channel_summary.to_csv(weekly_channel_csv_path, index=False)
    top_products_summary.to_csv(weekly_top_products_csv_path, index=False)
    daily_revenue.to_csv(weekly_daily_revenue_csv_path, index=False)
    anomaly_days.to_csv(weekly_anomaly_days_csv_path, index=False)

    print("Phase 4 completed successfully.\n")
    print("Saved files:")
    print("-", weekly_summary_json_path)
    print("-", weekly_channel_csv_path)
    print("-", weekly_top_products_csv_path)
    print("-", weekly_daily_revenue_csv_path)
    print("-", weekly_anomaly_days_csv_path)

except Exception as e:
    raise SystemExit(f"ERROR: Could not save Phase 4 outputs — {e}")

Phase 4 completed successfully.

Saved files:
- output\weekly_summary.json
- output\weekly_channel_breakdown.csv
- output\weekly_top_products.csv
- output\weekly_daily_revenue.csv
- output\weekly_anomaly_days.csv


In [39]:
summary = {
    "week_start": "2026-05-11",
    "week_end": "2026-05-17",
    "previous_week_start": "2026-05-04",
    "previous_week_end": "2026-05-10",
    "total_revenue": 446285.89,
    "previous_total_revenue": 477068.46,
    "revenue_pct_change": -6.45,
    "total_orders": 594,
    "previous_total_orders": 632,
    "orders_pct_change": -6.01,
    "aov": 751.32,
    "previous_aov": 754.86,
    "aov_pct_change": -0.47,
    "new_customers": 65,
    "returning_customers": 103,
    "pct_new_customers": 38.69,
    "pct_returning_customers": 61.31,
    "anomaly_days_count": 0
}

In [40]:
summary

{'week_start': '2026-05-11',
 'week_end': '2026-05-17',
 'previous_week_start': '2026-05-04',
 'previous_week_end': '2026-05-10',
 'total_revenue': 446285.89,
 'previous_total_revenue': 477068.46,
 'revenue_pct_change': -6.45,
 'total_orders': 594,
 'previous_total_orders': 632,
 'orders_pct_change': -6.01,
 'aov': 751.32,
 'previous_aov': 754.86,
 'aov_pct_change': -0.47,
 'new_customers': 65,
 'returning_customers': 103,
 'pct_new_customers': 38.69,
 'pct_returning_customers': 61.31,
 'anomaly_days_count': 0}

In [41]:
import json

with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("summary.json saved")

summary.json saved
